In [2]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", 
               "langchain", "langchain-groq", "langchain-community",
               "langgraph", "chromadb", "sentence-transformers", "pypdf"],
               capture_output=False)

CompletedProcess(args=['C:\\Users\\kanwa\\anaconda3\\python.exe', '-m', 'pip', 'install', 'langchain', 'langchain-groq', 'langchain-community', 'langgraph', 'chromadb', 'sentence-transformers', 'pypdf'], returncode=0)

In [4]:
import warnings
warnings.filterwarnings("ignore")

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from sentence_transformers import SentenceTransformer
import chromadb
from pypdf import PdfReader
import datetime

print("All imports successful")

All imports successful


In [5]:
llm = ChatGroq(
    api_key="Your_Grok_Key",
    model="llama-3.3-70b-versatile"
)
print("LLM ready")

LLM ready


In [6]:
# Read resume
reader = PdfReader("Neha_Resume.pdf")
full_text = "".join(page.extract_text() for page in reader.pages)
print(f"Resume loaded: {len(full_text)} characters")

# Chunk it
chunks = [full_text[i:i+350] for i in range(0, len(full_text), 350) 
          if full_text[i:i+350].strip()]
print(f"Split into {len(chunks)} chunks")

# Embed
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embed_model.encode(chunks).tolist()
print("Embeddings created")

# Store in vector DB
db = chromadb.Client()
collection = db.get_or_create_collection("neha_resume_day3")
collection.add(
    documents=chunks,
    embeddings=embeddings,
    ids=[f"c{i}" for i in range(len(chunks))]
)
print("RAG ready")

Resume loaded: 3799 characters
Split into 11 chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings created
RAG ready


In [10]:
@tool
def get_date() -> str:
    """Returns today's date. No input needed."""
    return str(datetime.date.today())

@tool
def calculator(expression: str) -> str:
    """Calculates math expression like 1234 * 5678"""
    try:
        return str(eval(expression))
    except:
        return "Invalid expression"

tools = [get_date, calculator]
print("Tools ready:", [t.name for t in tools])

Tools ready: ['get_date', 'calculator']


In [11]:
memory = MemorySaver()
config = {"configurable": {"thread_id": "neha_day3"}}

def get_resume_context(question):
    """Fetch relevant resume chunks for a question"""
    q_embed = embed_model.encode([question]).tolist()
    results = collection.query(query_embeddings=q_embed, n_results=3)
    return "\n".join(results['documents'][0])

def chat(user_input):
    try:
        # Check if question is about resume
        resume_keywords = ["internship", "project", "skill", "experience", 
                          "work", "built", "tech", "stack", "job", "iit",
                          "gradpipe", "isb", "neha", "resume", "suit", "role"]
        
        is_resume_question = any(
            kw in user_input.lower() for kw in resume_keywords
        )
        
        # If resume question → fetch context and inject into prompt
        if is_resume_question:
            context = get_resume_context(user_input)
            system_content = f"""You are an intelligent assistant for 
Neha Kanwadiya, IIT Bombay Engineering Physics student (2027).

Here is relevant information from her resume:
{context}

Answer the user's question using this resume information.
For date or math questions use your tools.
Be concise and professional."""
        else:
            system_content = """You are an intelligent assistant for 
Neha Kanwadiya, IIT Bombay Engineering Physics student (2027).
Be concise and professional."""

        agent = create_react_agent(
            llm,
            tools,
            checkpointer=memory,
            prompt=SystemMessage(content=system_content)
        )
        
        response = agent.invoke(
            {"messages": [HumanMessage(content=user_input)]},
            config=config
        )
        return response['messages'][-1].content
    
    except Exception as e:
        return f"Error: {str(e)}"

print("Agent ready")

Agent ready


In [12]:
print("=== RAG Test ===")
print(chat("What internships has Neha done?"))

print("\n=== Memory Test (follow-up) ===")
print(chat("Tell me more about the ISB one"))

print("\n=== Tool Test ===")
print(chat("What is today's date?"))

print("\n=== Reasoning Test ===")
print(chat("Based on her skills what jobs suit her?"))

=== RAG Test ===
Neha Kanwadiya has done the following internship: 
Software Engineering Intern at GradPipe, an early-stage startup building a talent marketplace connecting students with startups and global companies.

=== Memory Test (follow-up) ===
At ISB, Hyderabad, Neha worked on Executive Profiling and Behavior-based authorization for secure user login and protected content access. She implemented various features, including:

1. Video upload, playback, and adaptive streaming functionality with local storage & backend APIs.
2. Personalized user dashboard and video recommendation logic using React Hooks and Django ORM.

=== Tool Test ===
Today's date is 11 May 2026.

=== Reasoning Test ===
Based on Neha's skills in Large Language Models (LLMs), Prompt Engineering, and experience in software development, backend development, and AI-driven talent marketplace, some suitable job roles for her could be:

1. AI/ML Engineer
2. Software Developer
3. Backend Developer
4. Data Scientist
5. M

In [ ]:
print("Neha's AI Assistant — LangChain + RAG + Memory")
print("Type 'quit' to exit\n")

while True:
    user = input("You: ")
    if user.lower() == 'quit':
        print("Exiting.")
        break
    print(f"\nAI: {chat(user)}\n")
    print("─" * 50 + "\n")

Neha's AI Assistant — LangChain + RAG + Memory
Type 'quit' to exit



You:  What is her strongest technical skill?



AI: Based on the provided information, Neha's strongest technical skill appears to be Large Language Models (LLMs) and Prompt Engineering, as it is explicitly mentioned in the "Technologies" section of her resume.

──────────────────────────────────────────────────



You:  Tell me more about that



AI: Neha has expertise in designing and optimizing prompts for Large Language Models (LLMs) to achieve specific tasks and outputs. She has experience with various LLM architectures and has worked on fine-tuning and adapting these models for specific applications. Her skills in Prompt Engineering include:

1. Prompt design: Crafting effective prompts that elicit desired responses from LLMs.
2. Prompt optimization: Refining prompts to improve model performance, accuracy, and efficiency.
3. LLM fine-tuning: Adjusting model parameters to adapt to specific tasks, datasets, or domains.
4. Model evaluation: Assessing LLM performance, identifying biases, and mitigating errors.

Neha's strong foundation in LLMs and Prompt Engineering enables her to develop innovative solutions that leverage the capabilities of these models, such as text generation, language translation, and conversational AI.

──────────────────────────────────────────────────



...


AI: Based on the provided information, there is no direct evidence of Neha's experience or skills in fintech. Her background is in software development, AI, and engineering physics, but there is no mention of fintech-specific skills or experience. However, her strong foundation in technology and problem-solving could be transferable to a fintech role, and she may be able to adapt to the industry with additional training or experience.

──────────────────────────────────────────────────



You:  If she continues her carrier in both analyst and developer, then predicty if she is suitable for fintech job?



AI: If Neha continues to develop her skills in both analysis and development, she may be suitable for a fintech job. Her analytical skills, combined with her technical expertise in software development and AI, could be valuable in fintech roles such as:

1. Quantitative Analyst: She could apply her analytical skills to analyze financial data and develop models for risk management, portfolio optimization, and algorithmic trading.
2. Financial Data Scientist: Her experience in data analysis and machine learning could be applied to develop predictive models for financial forecasting, credit risk assessment, and fraud detection.
3. Fintech Product Developer: With her software development skills, she could design and develop fintech products such as mobile payment apps, digital wallets, or investment platforms.
4. Business Intelligence Developer: She could use her analytical and technical skills to develop business intelligence solutions for financial institutions, such as data visualizati

You:  What skills should have gained for this job?



AI: To be suitable for a fintech job, Neha should consider gaining skills in the following areas:

1. Programming languages:
	* Python (for data analysis, machine learning, and automation)
	* Java or C++ (for developing high-performance trading systems)
	* JavaScript (for web development and front-end applications)
2. Data analysis and science:
	* Data visualization tools (e.g., Tableau, Power BI)
	* Machine learning libraries (e.g., scikit-learn, TensorFlow)
	* Statistical modeling and data mining techniques
3. Financial knowledge:
	* Financial markets and instruments (e.g., stocks, bonds, derivatives)
	* Financial regulations and compliance (e.g., Dodd-Frank, GDPR)
	* Accounting and financial statement analysis
4. Fintech-specific skills:
	* Blockchain and distributed ledger technology
	* Cryptocurrencies and digital assets
	* Payment systems and gateways (e.g., PayPal, Stripe)
5. Cloud computing and infrastructure:
	* Cloud platforms (e.g., AWS, Azure, Google Cloud)
	* Containeriza